In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from models.wave2vec2.audio import to_tensors

/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-12-05 22:31:44.495831: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-05 22:31:44.498526: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-12-05 22:31:44.498536: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [2]:
import os
import pickle

INTERVAL_LENGTH = 15  # ms

class AudioSegmentDataset(Dataset):
    def __init__(self, label_dir, input_dir, cache_dir, num_samples=None):
        self.label_dir = label_dir
        self.input_dir = input_dir
        self.cache_dir = cache_dir

        # Get all CSV files
        self.csv_files = os.listdir(label_dir)
        if num_samples:
            self.csv_files = self.csv_files[:num_samples]

        # Create the cache directory if it doesn't exist
        if not os.path.exists(cache_dir):
            os.makedirs(cache_dir)

    def _load_audio_features(self, file_name):
        cache_file = os.path.join(self.cache_dir, f"{file_name}.pkl")
        if os.path.exists(cache_file):
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
        else:
            audio_file = os.path.join(self.input_dir, f"{file_name}.mp3")
            feature_vectors = to_tensors(audio_file, segment_length_ms=INTERVAL_LENGTH)
            with open(cache_file, 'wb') as f:
                pickle.dump(feature_vectors, f)
            return feature_vectors

    def __len__(self):
        return len(self.csv_files)

    def __getitem__(self, idx):
        file = self.csv_files[idx]
        file_path = os.path.join(self.label_dir, file)
        file_name = os.path.basename(file).split(".")[0]

        # Load labels
        df = pd.read_csv(file_path)

        # Load audio features from cache
        feature_vectors = self._load_audio_features(file_name)

        # Store features and labels
        return {
            'features': feature_vectors,
            'breaks': torch.tensor(df['break'].values, dtype=torch.float32),
            'start_times': df['start'].values,
            'end_times': df['end'].values
        }

def create_dataloaders(label_dir, input_dir, cache_dir, num_samples=None, batch_size=1):
    dataset = AudioSegmentDataset(label_dir, input_dir, cache_dir, num_samples)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # train/test/validate split (90/5/5) split
    train_size = int(0.7 * len(dataloader))
    test_size = int(0.1 * len(dataloader))
    val_size = len(dataloader) - train_size - test_size
    
    train_dataloader, val_dataloader, test_dataloader = torch.utils.data.random_split(
        dataset, [train_size, val_size, test_size]
    )
    
    print(f"Train size: {len(train_dataloader)}")
    print(f"Val size: {len(val_dataloader)}")
    print(f"Test size: {len(test_dataloader)}")
    
    return train_dataloader, val_dataloader, test_dataloader


In [3]:
data_dir = '../../data'
label_dir = f'{data_dir}/clean/segment_breaks'
input_dir = f'{data_dir}/clean/audio/vocals'
cache_dir = f'cache'

train_dataloader, val_dataloader, test_dataloader = create_dataloaders(
    label_dir=label_dir,
    input_dir=input_dir,
    cache_dir=cache_dir,
    num_samples=None,  # if None, all samples will be used
    batch_size=1
)

Train size: 46
Val size: 14
Test size: 6


In [4]:
# print information about all the samples
for sample in train_dataloader:
    print("===============================")
    print(f"Sample features shape: {sample['features'].shape}")
    print(f"Sample breaks shape: {sample['breaks'].shape}")

/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Sample features shape: torch.Size([16416, 768])
Sample breaks shape: torch.Size([16419])


/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Sample features shape: torch.Size([20330, 768])
Sample breaks shape: torch.Size([20332])


/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


...

Sample features shape: torch.Size([17101, 768])
Sample breaks shape: torch.Size([17104])


/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Sample features shape: torch.Size([14703, 768])
Sample breaks shape: torch.Size([14705])


/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Sample features shape: torch.Size([19936, 768])
Sample breaks shape: torch.Size([19939])


In [5]:
# print information about all the samples
for sample in test_dataloader:
    print("===============================")
    print(f"Sample features shape: {sample['features'].shape}")
    print(f"Sample breaks shape: {sample['breaks'].shape}")

/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Sample features shape: torch.Size([13713, 768])
Sample breaks shape: torch.Size([13716])


/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Sample features shape: torch.Size([7318, 768])
Sample breaks shape: torch.Size([7321])


/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


...

Sample features shape: torch.Size([14030, 768])
Sample breaks shape: torch.Size([14033])


/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Sample features shape: torch.Size([20333, 768])
Sample breaks shape: torch.Size([20336])


/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/transformers/configuration_utils.py:306: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Sample features shape: torch.Size([19105, 768])
Sample breaks shape: torch.Size([19108])


In [6]:
class RNNModel(nn.Module):
    def __init__(self, input_size=768, hidden_size=256, num_layers=1, dropout=0, bidirectional=False):
        super(RNNModel, self).__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout, bidirectional=bidirectional)
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), 1)

    def forward(self, x):
        directions = 2 if self.rnn.bidirectional else 1
        h0 = torch.zeros(directions, x.size(0), self.rnn.hidden_size).to(x.device)
        c0 = torch.zeros(directions, x.size(0), self.rnn.hidden_size).to(x.device)
        out, _ = self.rnn(x.unsqueeze(1), (h0, c0))
        out = self.fc(out)
        out = torch.sigmoid(out)
        return out

In [7]:
import torch
import torch.nn.functional as F

class CustomLoss(torch.nn.Module):
    def __init__(self, alpha=0.5, beta=1.0):
        """
        Custom loss function combining BCE and proximity loss
        
        Args:
        - alpha (float): Weight for BCE loss (0 <= alpha <= 1)
        - beta (float): Proximity loss sensitivity parameter
        """
        super(CustomLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, y_pred, y_true):
        """
        Compute custom loss
        
        Args:
        - y_pred (torch.Tensor): Predicted break probabilities (batch x time_steps)
        - y_true (torch.Tensor): Ground truth break labels (batch x time_steps)
        
        Returns:
        - torch.Tensor: Computed loss
        """
        # BCE Loss
        f_bce = F.binary_cross_entropy(y_pred, y_true, reduction='sum')

        # Proximity Loss
        #f_proximity = -self._compute_proximity_loss(y_pred, y_true)  # TODO: uncomment this line

        # Total Loss
        f_total = self.alpha * f_bce #+ (1 - self.alpha) * f_proximity  # TODO: uncomment this line

        return f_total

    def _compute_proximity_loss(self, y_pred, y_true):
        """
        Compute proximity loss
        
        Args:
        - y_pred (torch.Tensor): Predicted break probabilities
        - y_true (torch.Tensor): Ground truth break labels
        
        Returns:
        - torch.Tensor: Proximity loss
        """
        # Find indices of predicted and true breaks
        pred_breaks = torch.where(y_pred > 0.5)[0]
        true_breaks = torch.where(y_true > 0.5)[0]

        # If no breaks predicted or no true breaks, return 0
        if len(pred_breaks) == 0 or len(true_breaks) == 0:
            return torch.tensor(0.0, device=y_pred.device)

        # Compute proximity loss
        proximity_loss = 0.0
        for t in pred_breaks:
            # Find closest true break
            distances = torch.abs(t - true_breaks)
            min_distance = torch.min(distances)

            # Compute proximity term
            proximity_term = torch.exp(self.beta * min_distance) - 1
            proximity_loss += proximity_term

        return proximity_loss


In [8]:
def adjust_size(y_pred, y_true):
    size_diff = y_true.size(0) - y_pred.size(0)
    if size_diff > 0:
        y_true = y_true.squeeze()[:-size_diff]
    else:
        y_true = y_true.squeeze()
    y_pred = y_pred.squeeze()
    return y_pred, y_true

In [15]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            features = batch['features']
            targets = batch['breaks']

            optimizer.zero_grad()
            outputs = model(features)

            # Compute loss using custom loss function
            loss = criterion(*adjust_size(outputs, targets))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_loss:.4f}")

        model.eval()
        total_val_loss = 0
        
        accuracy_true = 0
        accuracy_false = 0
        
        with torch.no_grad():
            for batch in val_loader:
                features = batch['features']
                targets = batch['breaks']

                outputs = model(features)

                # Compute loss using custom loss function
                loss = criterion(*adjust_size(outputs, targets))
                total_val_loss += loss.item()
                
                # accuracy per class
                outputs, targets = adjust_size(outputs, targets)
                for i in range(len(targets)):
                    if outputs[i] > 0.5 and targets[i] == 1:
                        accuracy_true += 1
                    elif outputs[i] <= 0.5 and targets[i] == 0:
                        accuracy_false += 1

        avg_val_loss = total_val_loss / len(val_loader)
        print(f"Epoch {epoch+1}/{epochs}, Val Loss: {avg_val_loss:.4f}")
        # accuracy per class
        print(f"Accuracy True: {accuracy_true / len(val_loader.dataset)}")
        print(f"Accuracy False: {accuracy_false / len(val_loader.dataset)}")


In [16]:
print(f"Initializing the model")
model = RNNModel()
# criterion = torch.nn.BCELoss()
criterion = CustomLoss(alpha=0.5, beta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Initializing the model


In [17]:
train_model(model, train_dataloader, val_dataloader, criterion, optimizer, epochs=50)

Epoch 1/50, Train Loss: 1348.0624
Epoch 1/50, Val Loss: 970.5148
Accuracy True: 0.0
Accuracy False: 3298.0151515151515
Epoch 2/50, Train Loss: 905.7581
Epoch 2/50, Val Loss: 966.2334
Accuracy True: 0.0
Accuracy False: 3298.0151515151515
Epoch 3/50, Train Loss: 902.4374
Epoch 3/50, Val Loss: 962.1468
Accuracy True: 0.0
Accuracy False: 3298.0151515151515
Epoch 4/50, Train Loss: 899.4160
Epoch 4/50, Val Loss: 959.1633
Accuracy True: 0.0
Accuracy False: 3298.0151515151515
Epoch 5/50, Train Loss: 896.9652
Epoch 5/50, Val Loss: 957.0765
Accuracy True: 0.0
Accuracy False: 3298.0151515151515
Epoch 6/50, Train Loss: 895.0533
Epoch 6/50, Val Loss: 955.5738
Accuracy True: 0.0
Accuracy False: 3298.0151515151515
Epoch 7/50, Train Loss: 893.5331
Epoch 7/50, Val Loss: 954.4019
Accuracy True: 0.0
Accuracy False: 3298.0151515151515
Epoch 8/50, Train Loss: 892.2896
Epoch 8/50, Val Loss: 953.4237
Accuracy True: 0.0
Accuracy False: 3298.0151515151515
Epoch 9/50, Train Loss: 891.2539
Epoch 9/50, Val Loss: 

In [18]:
torch.save(model.state_dict(), 'model.pth')

In [13]:
def evaluate_model(model, test_loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in test_loader:
            features = batch['features']
            targets = batch['breaks']

            outputs = model(features)

            # Compute loss using custom loss function
            loss = criterion(*adjust_size(outputs, targets))
            total_loss += loss.item()

    avg_loss = total_loss / len(test_loader)
    print(f"Test Loss: {avg_loss:.4f}")
    return avg_loss


In [14]:
evaluate_model(model, test_dataloader, criterion)

Test Loss: 1045.9485


1045.948481241862